In [1]:
from datetime import datetime
from pathlib import Path
import sys

sys.path.insert(0, str(Path("../..").resolve()))

import ad_safe_lib as ad_safe

print("challenge:", ad_safe.CHALLENGE_DIR)
print("device:", ad_safe.DEVICE)
print("dataset:", ad_safe.DATA_DIR)
examples_root = ad_safe.AD_SAFE_RUNS_DIR / "notebook_examples"
examples_root.mkdir(parents=True, exist_ok=True)


challenge: /home/madmazoku/project/ml-bootcamp-2026/challenge
device: cuda
dataset: /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset


# Teacher Then Student Distillation

Train a small pretrained teacher, warm up the full scratch student, then continue that student with cached teacher logits. The run is still notebook-sized, but it uses enough data and epochs for the student to learn before distillation.


In [4]:
run_id = "notebook-03-" + datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
output_dir = examples_root / run_id

# Slightly larger subset reduces variance while staying notebook-friendly.
train_source = ad_safe.DatasetSourceSpec("train", fraction=0.08, seed=30)
eval_sources = {
    "val": ad_safe.DatasetSourceSpec("val", fraction=0.10, seed=101),
    "test": ad_safe.DatasetSourceSpec("test", fraction=0.10, seed=102),
}
teacher_path = output_dir / "teacher_mobilenet.pt"
student_seed = 32

teacher_phase = ad_safe.PhaseSpec(
    job_index=0,
    phase_index=0,
    prefix="teacher_mobilenet",
    name="teacher",
    requested_seed=31,
    config=ad_safe.TrainingConfig(
        base_model="mobilenet_v3_small",
        epochs=3,
        patience=2,
        batch_size=16,
        learning_rate=(1e-3,),
        resplit_runs=1,
    ),
    model_filename=teacher_path.name,
    history_filename="teacher_mobilenet_history.png",
    json_filename="teacher_mobilenet.json",
)

student_warmup_phase = ad_safe.PhaseSpec(
    job_index=1,
    phase_index=0,
    prefix="student_warmup",
    name="warmup",
    requested_seed=student_seed,
    config=ad_safe.TrainingConfig(
        base_model="simple_cnn",
        epochs=4,
        patience=2,
        batch_size=16,
        learning_rate=(6e-4,),
        resplit_runs=1,
    ),
    unfreeze_all=True,
    model_filename="student_warmup.pt",
    history_filename="student_warmup_history.png",
    json_filename="student_warmup.json",
)

distilled_student_phase = ad_safe.PhaseSpec(
    job_index=1,
    phase_index=1,
    prefix="student_distilled",
    name="distilled",
    requested_seed=student_seed,
    config=ad_safe.TrainingConfig(
        base_model="simple_cnn",
        epochs=6,
        patience=3,
        batch_size=16,
        learning_rate=(2e-4,),
        resplit_runs=1,
        teacher_model_path=str(teacher_path),
        distillation_alpha=0.15,
        distillation_temperature=3.0,
    ),
    unfreeze_all=True,
    model_filename="student_distilled.pt",
    history_filename="student_distilled_history.png",
    json_filename="student_distilled.json",
)

plan = ad_safe.RunPlan(
    output_dir=output_dir,
    run_id=run_id,
    train_split="train",
    eval_splits=("val", "test"),
    jobs=(
        ad_safe.JobSpec(job_index=0, job_id="teacher", display_name="teacher", backbone="mobilenet_v3_small", phases=(teacher_phase,)),
        ad_safe.JobSpec(job_index=1, job_id="student", display_name="student", backbone="simple_cnn", phases=(student_warmup_phase, distilled_student_phase)),
    ),
    resume=False,
    force=True,
    setup_path=output_dir / "setup.json",
    check_foreign_contract=False,
    train_source=train_source,
    eval_sources=eval_sources,
)

result = ad_safe.run_training_plan(plan)
[(phase.prefix, phase.model_path) for phase in result.phase_results]


Using device: cuda
Run id: notebook-03-2026-04-22-16-08-46
Output dir: /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46
Train split: train
Eval splits: val, test
Resume: False
Force: True
Cooldown: {'every_epochs': 0, 'seconds': 0.0, 'gpu_max_temp': 0, 'gpu_resume_temp': 0, 'gpu_temp_check_seconds': 15.0}
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/train...
Using stratified dataset source train: 800/10000 samples (fraction=0.08, seed=30)
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/val...
Using stratified dataset source val: 1000/10000 samples (fraction=0.1, seed=101)
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/test...
Using stratified dataset source test: 1000/10000 samples (fraction=0.1, seed=102)
Setup saved to /hom

Evaluating (val):   0%|          | 0/5 [00:00<?, ?it/s]

Initial val metrics: acc=0.5000 auc=0.5088 nll=0.7154 conf=0.5929 margin=0.1858 wrong_conf=0.5908 safe_recall=0.5500 unsafe_recall=0.4500


Training:   0%|          | 0/43 [00:00<?, ?it/s]

Evaluating (val):   0%|          | 0/5 [00:00<?, ?it/s]

...

Split 1/1 | Epoch 5/6 | global_epoch=5 | train=0.3931 | val=0.6375 | auc=0.6931 | nll=0.7439 | distill_train=0.6781 | best=0.6625 @ epoch 2 | comparison=worse | time=3.36s
Early stopping at epoch 5 (patience=3)
Loading model from /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/student_distilled.pt...
Loaded best model from epoch 2 (score: 0.6625)
Saving model to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/student_distilled.pt...
Figure saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/student_distilled_history.png
Loading model from /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/student_distilled.pt...


Evaluating (val):   0%|          | 0/63 [00:00<?, ?it/s]

Evaluating (test):   0%|          | 0/63 [00:00<?, ?it/s]

JSON saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/student_distilled.json
Metrics CSV saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/accuracy.csv
Setup saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/setup.json

Results
prefix             val                                                                                             test                                                                                          
-----------------  ----------------------------------------------------------------------------------------------  ----------------------------------------------------------------------------------------------
student_distilled  acc=0.6400 auc=0.7181 nll=0.6189 avg_wrong_conf=0.6386 safe_recall=0.5940 unsafe_re

[('teacher_mobilenet',
  PosixPath('/home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/teacher_mobilenet.pt')),
 ('student_warmup',
  PosixPath('/home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/student_warmup.pt')),
 ('student_distilled',
  PosixPath('/home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/student_distilled.pt'))]

In [5]:
ad_safe.run_evaluation_plan(
    ad_safe.EvaluationPlan(
        models=tuple(
            ad_safe.ModelEvalSpec(path=phase.model_path, row_id=phase.prefix)
            for phase in result.phase_results
        ),
        datasets=(
            ad_safe.DatasetEvalSpec(name="val", batch_size=16, source=eval_sources["val"]),
            ad_safe.DatasetEvalSpec(name="test", batch_size=16, source=eval_sources["test"]),
        ),
        output_dir=output_dir,
        sort_key="acc_test",
        title="Teacher/student comparison",
    )
)


Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/val...
Using stratified dataset source val: 1000/10000 samples (fraction=0.1, seed=101)
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/test...
Using stratified dataset source test: 1000/10000 samples (fraction=0.1, seed=102)

Evaluating teacher_mobilenet.pt...
Loading model from /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/teacher_mobilenet.pt...


Evaluating (val):   0%|          | 0/63 [00:00<?, ?it/s]

val: acc=0.7960 auc=0.9027 nll=0.4589 conf=0.8797 margin=0.7594 wrong_conf=0.7609 safe_recall=0.7180 unsafe_recall=0.8740


Evaluating (test):   0%|          | 0/63 [00:00<?, ?it/s]

test: acc=0.7260 auc=0.8497 nll=0.6221 conf=0.8618 margin=0.7237 wrong_conf=0.7810 safe_recall=0.5760 unsafe_recall=0.8760

Evaluating student_warmup.pt...
Loading model from /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/student_warmup.pt...


...

Evaluating (val):   0%|          | 0/63 [00:00<?, ?it/s]

val: acc=0.6400 auc=0.7181 nll=0.6189 conf=0.6779 margin=0.3558 wrong_conf=0.6386 safe_recall=0.5940 unsafe_recall=0.6860


Evaluating (test):   0%|          | 0/63 [00:00<?, ?it/s]

test: acc=0.6130 auc=0.6588 nll=0.6722 conf=0.6748 margin=0.3496 wrong_conf=0.6541 safe_recall=0.5320 unsafe_recall=0.6940

Teacher/student comparison
model              val                                                                                             test                                                                                          
-----------------  ----------------------------------------------------------------------------------------------  ----------------------------------------------------------------------------------------------
teacher_mobilenet  acc=0.7960 auc=0.9027 nll=0.4589 avg_wrong_conf=0.7609 safe_recall=0.7180 unsafe_recall=0.8740  acc=0.7260 auc=0.8497 nll=0.6221 avg_wrong_conf=0.7810 safe_recall=0.5760 unsafe_recall=0.8760
student_distilled  acc=0.6400 auc=0.7181 nll=0.6189 avg_wrong_conf=0.6386 safe_recall=0.5940 unsafe_recall=0.6860  acc=0.6130 auc=0.6588 nll=0.6722 avg_wrong_conf=0.6541 safe_recall=0.5320 unsafe_recall=0.6940
student_w

EvaluationRunResult(rows=(MetricsMatrixRow(row_id='teacher_mobilenet', metrics_by_dataset={'val': ClassificationMetrics(accuracy=0.7960000038146973, auc=0.902724, nll=0.4588557183742523, avg_conf=0.87971431016922, avg_margin=0.7594285607337952, avg_correct_conf=0.7732724547386169, avg_wrong_conf=0.7608869075775146, safe_recall=0.7179999947547913, unsafe_recall=0.8740000128746033), 'test': ClassificationMetrics(accuracy=0.7260000109672546, auc=0.849744, nll=0.6221151947975159, avg_conf=0.861845850944519, avg_margin=0.7236917018890381, avg_correct_conf=0.7078830003738403, avg_wrong_conf=0.7809542417526245, safe_recall=0.5759999752044678, unsafe_recall=0.8759999871253967)}, metadata={'model_name': 'teacher_mobilenet.pt', 'model_path': '/home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-03-2026-04-22-16-08-46/teacher_mobilenet.pt', 'rank': 1}), MetricsMatrixRow(row_id='student_distilled', metrics_by_dataset={'val': ClassificationMetrics(acc